## Downscaling and moving windows
- Downscaling DTM rasters to 30m to have same-level scale with LST images 
- computing moving windows and study the correlations with LST depending on the size of the window.

### Libraries and general paths

In [ ]:
!pip install rasterio
!pip install tqdm

In [ ]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import numpy as np
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from scipy.ndimage import convolve
from tqdm import tqdm  
from rasterio.crs import CRS

In [ ]:
# user-defined paths
base_dir = '../../../datos-notebooks/1.global-data-cba/'
dtm_raster_name = 'fabdem_cba_30m.tif'
city_boundary = 'cba_boundary.gpkg'
landsat_image = 'LC08_L2SP_229082_20240125_20240130_02_T1'

In [ ]:
# derived paths

# LST
lst_cropped_path = os.path.join(base_dir, f'LST/LST_cropped_celsius{landsat_image}.tif') # values out of bound are 0
lst_clipped_path = os.path.join(base_dir, f'LST/LST_clipped_celsius{landsat_image}.tif') #values out of bound are nan

#inputs
dtm_path = os.path.join(base_dir, 'topography/', dtm_raster_name)
variables_30m_path = os.path.join(base_dir, 'topography/', 'rasters_30m/')
moving_windows_path = os.path.join(base_dir, 'topography/', 'moving_windows/')

bound = os.path.join(base_dir, 'vectors',city_boundary)
gdf = gpd.read_file(bound)

In [ ]:
def save_raster(filename, array, output_dir):
    profile = lst_profile.copy()
    profile.update(dtype=rasterio.float32, count=1, compress='lzw')
    with rasterio.open(os.path.join(output_dir, filename), 'w', **profile) as dst:
        dst.write(array.astype(np.float32), 1)

In [ ]:
with rasterio.open(lst_clipped_path) as lst:
    lst_clip = lst.read(1)          
    lst_transform = lst.transform 
    lst_profile = lst.profile
    lst_crs = lst.crs
    lst_shape = (lst.height, lst.width)
    
valid_mask = np.ones(lst_shape, dtype=bool)  # Initialize valid mask

### align with LST raster

#### align

In [ ]:
os.makedirs(variables_30m_path, exist_ok=True)
dtm_resampled_path = os.path.join(variables_30m_path, 'dtm_30m.tif')

In [ ]:
with rasterio.open(dtm_path) as dtm_src:
    dtm_data = dtm_src.read(1)
    dtm_profile = dtm_src.profile
    nodata_val = dtm_src.nodata  

    # Force CRS if missing
    if dtm_src.crs is None:
        dtm_crs = CRS.from_epsg(7791)  
    else:
        dtm_crs = dtm_src.crs

    # Create output array
    dtm_resampled = np.empty(lst_shape, dtype=np.float32)
    
    # mask nodata values for rescaling
    if nodata_val is not None:
        dtm_masked = np.where(dtm_data == nodata_val, np.nan, dtm_data)
    else:
        dtm_masked = dtm_data.astype(np.float32)

    # Reproject / downscale using the average
    reproject(
        source=dtm_masked,
        destination=dtm_resampled,
        src_transform=dtm_src.transform,
        src_crs=dtm_crs,
        dst_transform=lst_transform,
        dst_crs=lst_crs,
        resampling=Resampling.average
    )

save_raster('dtm_30m.tif', dtm_resampled, variables_30m_path)

In [ ]:
with rasterio.open(dtm_resampled_path) as src:
    dtm_resampled = src.read(1)
    
fig, ax = plt.subplots(figsize=(10,10))
img = ax.imshow(dtm_resampled, cmap='YlGn') 
ax.set_title("DTM 30m")
plt.colorbar(img, ax=ax, label="terrain height")
plt.show()


#### extract:
- mean elevation
- mean slope
- aspect
- northness
- eastness


In [ ]:
# Compute slope and aspect

# Pixel size (xres, yres)
xres = lst_transform.a
yres = -lst_transform.e  

# Compute gradients
dz_dy, dz_dx = np.gradient(dtm_resampled, yres, xres)

# Remove padding
dz_dy = dz_dy[1:-1, 1:-1]
dz_dx = dz_dx[1:-1, 1:-1]

# Slope in radians
slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))

# Aspect (0 = North)
aspect = np.arctan2(dz_dx, -dz_dy)
aspect = np.where(aspect < 0, 2 * np.pi + aspect, aspect)

# Compute northness and eastness
northness = np.cos(aspect)
eastness = np.sin(aspect)

save_raster('elev_mean.tif', dtm_resampled, variables_30m_path)
save_raster('slope_mean.tif', np.degrees(slope), variables_30m_path)  # degrees
save_raster('northness.tif', northness, variables_30m_path)
save_raster('eastness.tif', eastness, variables_30m_path)

print("Finished. Files saved as elev_mean.tif, slope_mean.tif, northness.tif, eastness.tif")

In [ ]:
# plot
files = {
    "Elevación media (m)": os.path.join(variables_30m_path, "elev_mean.tif"),
    "Pendiente media (°)": os.path.join(variables_30m_path, "slope_mean.tif"),
    "Orientación norte": os.path.join(variables_30m_path, "northness.tif"),
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))


for ax, (title, path) in zip(axes, files.items()):
    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")

    # apply mask keeping the 2D shape
    data = np.where(valid_mask, data, np.nan)

    cmap = 'terrain' if 'elevation' in title.lower() else 'RdYlBu_r'

    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

### Moving windows

In [ ]:
files = {
    "elev_mean": os.path.join(variables_30m_path, "elev_mean.tif"),
    "slope_mean": os.path.join(variables_30m_path, "slope_mean.tif"),
    "northness": os.path.join(variables_30m_path, "northness.tif"),
    "eastness": os.path.join(variables_30m_path, "eastness.tif"),
} 


In [ ]:
def distance_weighted_kernel(size):
    """Create a distance-weighted Gaussian kernel."""
    assert size % 2 == 1 #Size must be odd
    center = size // 2
    y, x = np.ogrid[:size, :size]
    distance = np.sqrt((x - center)**2 + (y - center)**2)
    sigma = size / 4
    weights = np.exp(-(distance**2) / (2 * sigma**2))
    weights /= weights.sum()  # normalize
    return weights

def moving_window_weighted(array, kernel):
    """Apply convolution with given kernel."""
    return convolve(array, kernel, mode='nearest')


In [ ]:
# visualyze weighted kernel
k = distance_weighted_kernel(11)

plt.imshow(k, cmap='viridis')
plt.colorbar(label="Weight")
plt.title("Distance-weighted Gaussian kernel (11x11)")
plt.show()

In [ ]:
# visualyze weight profile
plt.plot(k[k.shape[0]//2], label="Center row")
plt.plot(k[:, k.shape[1]//2], label="Center column")
plt.legend()
plt.title("Weights decay from center")
plt.show()

In [ ]:
os.makedirs(moving_windows_path, exist_ok=True)

max_window = 151

for var_name, path in tqdm(files.items(), desc="Variables"):  # Outer progress
    var_dir = os.path.join(moving_windows_path, var_name)
    os.makedirs(var_dir, exist_ok=True)

    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)

    for w in tqdm(range(1, max_window+1, 2), desc=f"Windows {var_name}", leave=False):
        kernel = distance_weighted_kernel(w)
        avg_window = moving_window_weighted(data, kernel)

        out_name = f"{var_name}_win{w}.tif"
        save_raster(out_name, avg_window, var_dir)

print("Finished. All variables saved with distance-weighted moving windows.") #takes 50' aprox


In [ ]:
# Plots
window_sizes = range(1, 21,2)  
variables = ["elev_mean", "slope_mean", "northness"]#, "eastness"]

for var_name in variables:
    var_folder = os.path.join(moving_windows_path, var_name)
 
    # Create subplot grid
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()

    for idx, w in enumerate(window_sizes):
        raster_path = os.path.join(var_folder, f"{var_name}_win{w}.tif")
        with rasterio.open(raster_path) as src_var:
            var_data = src_var.read(1).astype(float)

        # Plot
        im = axes[idx].imshow(var_data, cmap='terrain' if 'elev' in var_name else 'RdYlBu_r')
        axes[idx].set_title(f"Win {w}")
        axes[idx].axis('off')
        plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

    plt.suptitle(f"{var_name} - Windows 1 to 10", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


### correlations with LST

In [ ]:
window_sizes = range(1, 151, 2)

variables = {
    "elev_mean": "Elevación media",
    "slope_mean": "Pendiente media",
    "northness": "Orientación norte",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()

for ax, (var_name, label) in zip(axes, variables.items()):
    var_folder = os.path.join(moving_windows_path, var_name)
    correlations = []

    for w in window_sizes:
        raster_path = os.path.join(var_folder, f"{var_name}_win{w}.tif")
        with rasterio.open(raster_path) as src_var:
            var_data = src_var.read(1).astype(float)

        valid = ~np.isnan(lst_clip)
        corr = np.corrcoef(var_data[valid], lst_clip[valid])[0, 1]
        correlations.append(corr)

    ax.plot(window_sizes, correlations, marker='o', markersize=3)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Tamaño de ventana (píxeles)")
    ax.set_ylabel("Correlación de Pearson $r$")
    ax.grid(True)



plt.tight_layout()
plt.show()